In [4]:
console.log("Hello World")


Hello World


In [7]:
const token = Deno.env.get("GITHUB_TOKEN");
if (!token) throw new Error("GITHUB_TOKEN environment variable is not set.");

const apiBase = Deno.env.get("GITHUB_MODELS_URL") ?? "https://models.github.ai";
const endpoint = `${apiBase}/inference/chat/completions`;

const model = Deno.env.get("GITHUB_MODEL") ?? "openai/gpt-4.1";
const question = "What is the capital of France?";

const payload = {
  model,
  messages: [{ role: "user", content: question }]
};

const resp = await fetch(endpoint, {
  method: "POST",
  headers: {
    "Authorization": `Bearer ${token}`,
    "Accept": "application/json",
    "Content-Type": "application/json",
    "X-GitHub-Api-Version": "2022-11-28"
  },
  body: JSON.stringify(payload),
});

if (!resp.ok) {
  const text = await resp.text();
  console.error(`Request failed (${resp.status}): ${text}`);
} else {
  const json = await resp.json();
  const reply =
    json.choices?.[0]?.message?.content ??
    json.choices?.[0]?.content ??
    json.output_text ??
    JSON.stringify(json, null, 2);
  console.log("Model reply:\n", reply);
}

Model reply:
 The capital of France is **Paris**.


In [ ]:
// lc-github-models.ts
import { RunnableLambda } from "npm:@langchain/core/runnables";
import { AIMessage } from "npm:@langchain/core/messages";

function lcToGhMessages(input: any): Array<{ role: string; content: string }> {
  // 1) If the caller passed { messages }, use that; else { question }
  let msgs = input?.messages;
  if (!msgs && input?.question) {
    msgs = [{ role: "user", content: String(input.question) }];
  }

  // 2) If a prompt piped LC BaseMessages directly, input itself may be an array
  if (!msgs && Array.isArray(input)) {
    msgs = input;
  }

  if (!Array.isArray(msgs) || msgs.length === 0) {
    throw new Error("Provide { messages } or { question }.");
  }

  const mapRole = (t: string | undefined) => {
    const r = (t ?? "").toLowerCase();
    if (r === "human" || r === "user") return "user";
    if (r === "ai" || r === "assistant") return "assistant";
    if (r === "system") return "system";
    if (r === "developer") return "developer";
    // Fallback: if OpenAI-style already provided
    if (r === "tool" || r === "function") return "assistant"; // conservative fallback
    return "user";
  };

  const toText = (content: any): string => {
    if (typeof content === "string") return content;
    if (Array.isArray(content)) {
      // LC content parts: [{type:"text", text:"..."}, ...]
      return content
        .map((p) =>
          typeof p === "string"
            ? p
            : p?.text ?? p?.content ?? JSON.stringify(p)
        )
        .join("");
    }
    // Some LC BaseMessage store in .content/.lc_kwargs/etc.
    return content?.text ?? content?.content ?? String(content ?? "");
  };

  return msgs.map((m: any) => {
    const typeHint =
      m.role ?? m._type ?? m.type ?? m._getType?.() ?? m.constructor?.name;
    return {
      role: mapRole(typeHint),
      content: toText(m.content),
    };
  });
}

export function GitHubModelsChat(options?: {
  base?: string;
  org?: string;
  model?: string;
  token?: string | null;
  apiVersion?: string;
}) {
  const base = options?.base ?? Deno.env.get("GITHUB_MODELS_URL") ?? "https://models.github.ai";
  const org  = options?.org  ?? Deno.env.get("GITHUB_ORG") ?? undefined;
  const model= options?.model?? Deno.env.get("GITHUB_MODEL") ?? "openai/gpt-4.1";
  const token= options?.token?? Deno.env.get("GITHUB_TOKEN");
  const apiVersion = options?.apiVersion ?? "2022-11-28";

  if (!token) throw new Error("GITHUB_TOKEN is not set.");

  const endpoint = org
    ? `${base}/orgs/${org}/inference/chat/completions`
    : `${base}/inference/chat/completions`;

  return new RunnableLambda({
    name: "GitHubModelsChat",
    func: async (input: any) => {
      const messages = lcToGhMessages(input);

      const resp = await fetch(endpoint, {
        method: "POST",
        headers: {
          "Authorization": `Bearer ${token}`,
          "Accept": "application/json",
          "Content-Type": "application/json",
          "X-GitHub-Api-Version": apiVersion
        },
        body: JSON.stringify({ model, messages }),
      });

      if (!resp.ok) {
        const text = await resp.text();
        throw new Error(`GitHub Models request failed (${resp.status}): ${text}`);
      }

      const json = await resp.json();
      const reply =
        json.choices?.[0]?.message?.content ??
        json.choices?.[0]?.content ??
        json.output_text ??
        JSON.stringify(json, null, 2);

      return new AIMessage(reply);
    }
  });
}


In [25]:
import { ChatPromptTemplate } from "npm:@langchain/core/prompts";
import { StringOutputParser } from "npm:@langchain/core/output_parsers";
// import { GitHubModelsChat } from "./lc-github-models.ts";

const prompt = ChatPromptTemplate.fromMessages([
  ["system", "You are concise and accurate."],
  ["human", "{question}"],
]);

const model = GitHubModelsChat(); // uses env: GITHUB_TOKEN, GITHUB_MODEL, etc.
const toText = new StringOutputParser();

const chain = prompt.pipe(model).pipe(toText);

const answer = await chain.invoke({ question: "What is the capital of France?" });
console.log(answer); // "Paris"


The capital of France is Paris.


In [24]:
// hello_github_models_deno.ts
import { generateText } from "npm:ai";
import { createOpenAICompatible } from "npm:@ai-sdk/openai-compatible";

const token = Deno.env.get("GITHUB_TOKEN");
if (!token) throw new Error("GITHUB_TOKEN is not set.");

const github = createOpenAICompatible({
  name: "github",
  baseURL: "https://models.github.ai/inference", // GitHub Models endpoint
  headers: {
    Authorization: `Bearer ${token}`,
    "X-GitHub-Api-Version": "2022-11-28",
  },
});

const { text } = await generateText({
  model: github("openai/gpt-4o"), // or "openai/gpt-4.1", etc.
  prompt: "Say hello from GitHub Models in Deno!",
});

console.log(text);


Hello from GitHub Models in Deno! 🎉 🚀  
If you're running Deno with GitHub integrations, you're diving into some cutting-edge development — keep on building amazing things! 🌟
